# Crear y conectar el Deployment Job

Ejecutar una vez por workspace con una identidad de administración.

In [ ]:
import os

from databricks.sdk import WorkspaceClient
from mlflow.tracking import MlflowClient

from iris_mlflow_utils import build_deployment_config

config = build_deployment_config()
workspace = WorkspaceClient()
notebook_root = config.notebook_root.rstrip('/')
cluster_id = os.getenv('IRIS_DEPLOYMENT_CLUSTER_ID', '').strip()
if not cluster_id:
    raise ValueError('Configura IRIS_DEPLOYMENT_CLUSTER_ID para crear el job.')
service_principal = os.getenv('IRIS_DEPLOYMENT_SERVICE_PRINCIPAL', '').strip()
job_name = config.job_name
tasks = [
    {
        'task_key': 'evaluate_model',
        'existing_cluster_id': cluster_id,
        'notebook_task': {'notebook_path': f'{notebook_root}/evaluate_model'},
    },
    {
        'task_key': 'Approval_Check',
        'existing_cluster_id': cluster_id,
        'depends_on': [{'task_key': 'evaluate_model'}],
        'notebook_task': {'notebook_path': f'{notebook_root}/approval'},
        'max_retries': 0,
    },
    {
        'task_key': 'deploy_model',
        'existing_cluster_id': cluster_id,
        'depends_on': [{'task_key': 'Approval_Check'}],
        'notebook_task': {'notebook_path': f'{notebook_root}/deploy_model'},
    },
]
settings = {
    'name': job_name,
    'max_concurrent_runs': 1,
    'tasks': tasks,
    'parameters': [
        {'name': 'model_name', 'default': config.model_name},
        {'name': 'model_version', 'default': ''},
    ],
}
if service_principal:
    settings['run_as'] = {'service_principal_name': service_principal}
created_job = workspace.api_client.do('POST', '/api/2.1/jobs/create', body=settings)
job_id = created_job['job_id']
registry = MlflowClient(registry_uri='databricks-uc')
registry.update_registered_model(name=config.model_name, deployment_job_id=str(job_id))
print({'job_id': job_id, 'model_name': config.model_name, 'endpoint': config.endpoint_name})
